In [75]:
import numpy as np
import pandas as pd
import folium
import branca.colormap as cm

# naszym celem jest sprawdzenie jak rozmeszczenie tras obieranych przez uzytkownikow wypozyczalni zmienialo sie na przestrzeni roku 2025
# pokazemy to na podstawie mapy - zaznaczymy na mapie stacje i polaczymy je tworzac graf
# im ciemniejsza "kreska" laczace wierzcholki tym czesciej zostalo wybierane to polaczenie
# wybierzemy 4 miesiace reprezentujace dana pore roku:
# zima - styzen 2025
# wiosna - kwiecien 2025
# lato - lipiec 2025
# jesien - pazdziernik 2025

# na poczatku musimy przygotowac dane
# dla kazdej pory roku policzymy ilosc tras (nie rozrozniamy kierunkow tras) i wybierzemy 100 najczestszych
# ZIMA
styczen = pd.read_csv('DANE/JC-202501-citibike-tripdata.csv')
trasy_styczen = styczen.groupby(['start_station_name', 'end_station_name']).size().reset_index(name='liczba_tras')
trasy_styczen = trasy_styczen.sort_values(by='liczba_tras', ascending=False)
trasy_styczen['stacja_1'] = trasy_styczen[['start_station_name', 'end_station_name']].min(axis=1)
trasy_styczen['stacja_2'] = trasy_styczen[['start_station_name', 'end_station_name']].max(axis=1)
trasy_styczen = trasy_styczen.groupby(['stacja_1', 'stacja_2'])['liczba_tras'].sum().reset_index()
trasy_styczen = trasy_styczen.rename(columns={'stacja_1': 'start_station_name', 'stacja_2': 'end_station_name'})
trasy_styczen = trasy_styczen.sort_values(by='liczba_tras', ascending=False).head(100).reset_index(drop=True)
# WIOSNA 
kwiecien = pd.read_csv('DANE/JC-202504-citibike-tripdata.csv')
trasy_kwiecien = kwiecien.groupby(['start_station_name', 'end_station_name']).size().reset_index(name='liczba_tras')
trasy_kwiecien = trasy_kwiecien.sort_values(by='liczba_tras', ascending=False)
trasy_kwiecien['stacja_1'] = trasy_kwiecien[['start_station_name', 'end_station_name']].min(axis=1)
trasy_kwiecien['stacja_2'] = trasy_kwiecien[['start_station_name', 'end_station_name']].max(axis=1)
trasy_kwiecien = trasy_kwiecien.groupby(['stacja_1', 'stacja_2'])['liczba_tras'].sum().reset_index()
trasy_kwiecien = trasy_kwiecien.rename(columns={'stacja_1': 'start_station_name', 'stacja_2': 'end_station_name'})
trasy_kwiecien = trasy_kwiecien.sort_values(by='liczba_tras', ascending=False).head(100).reset_index(drop=True)
# LATO 
lipiec = pd.read_csv('DANE/JC-202507-citibike-tripdata.csv')
trasy_lipiec = lipiec.groupby(['start_station_name', 'end_station_name']).size().reset_index(name='liczba_tras')
trasy_lipiec = trasy_lipiec.sort_values(by='liczba_tras', ascending=False)
trasy_lipiec['stacja_1'] = trasy_lipiec[['start_station_name', 'end_station_name']].min(axis=1)
trasy_lipiec['stacja_2'] = trasy_lipiec[['start_station_name', 'end_station_name']].max(axis=1)
trasy_lipiec = trasy_lipiec.groupby(['stacja_1', 'stacja_2'])['liczba_tras'].sum().reset_index()
trasy_lipiec = trasy_lipiec.rename(columns={'stacja_1': 'start_station_name', 'stacja_2': 'end_station_name'})
trasy_lipiec = trasy_lipiec.sort_values(by='liczba_tras', ascending=False).head(100).reset_index(drop=True)
# JESIEN
pazdziernik = pd.read_csv('DANE/JC-202510-citibike-tripdata.csv')
trasy_pazdziernik = pazdziernik.groupby(['start_station_name', 'end_station_name']).size().reset_index(name='liczba_tras')
trasy_pazdziernik = trasy_pazdziernik.sort_values(by='liczba_tras', ascending=False)
trasy_pazdziernik['stacja_1'] = trasy_pazdziernik[['start_station_name', 'end_station_name']].min(axis=1)
trasy_pazdziernik['stacja_2'] = trasy_pazdziernik[['start_station_name', 'end_station_name']].max(axis=1)
trasy_pazdziernik = trasy_pazdziernik.groupby(['stacja_1', 'stacja_2'])['liczba_tras'].sum().reset_index()
trasy_pazdziernik = trasy_pazdziernik.rename(columns={'stacja_1': 'start_station_name', 'stacja_2': 'end_station_name'})
trasy_pazdziernik = trasy_pazdziernik.sort_values(by='liczba_tras', ascending=False).head(100).reset_index(drop=True)

# teraz wybierzemy koordynaty unikalnych stacji
wszystkie_miesiace = pd.concat([styczen, kwiecien, lipiec, pazdziernik], ignore_index=True)
stacje_start = wszystkie_miesiace[['start_station_name', 'start_lat', 'start_lng']].copy()
stacje_start.columns = ['nazwa_stacji', 'szerokosc_geo', 'dlugosc_geo']
stacje_end = wszystkie_miesiace[['end_station_name', 'end_lat', 'end_lng']].copy()
stacje_end.columns = ['nazwa_stacji', 'szerokosc_geo', 'dlugosc_geo']
stacje = pd.concat([stacje_start, stacje_end])
stacje = wszystkie_stacje.dropna(subset=['nazwa_stacji']) \
                                  .drop_duplicates(subset=['nazwa_stacji']) \
                                  .reset_index(drop=True)
trasy_pazdziernik
# przygotowalismy dane, mozemy teraz przejsc do rysowania map dla poszczegolnych miesiecy
# zapiszmy je jeszcze do pliku
trasy_styczen.to_csv('WYNIKI/POMOCNICZE_TABELE/trasy_styczen.csv', index=False)
trasy_kwiecien.to_csv('WYNIKI/POMOCNICZE_TABELE/trasy_kwiecien.csv', index=False)
trasy_lipiec.to_csv('WYNIKI/POMOCNICZE_TABELE/trasy_lipiec.csv', index=False)
trasy_pazdziernik.to_csv('WYNIKI/POMOCNICZE_TABELE/trasy_pazdziernik.csv', index=False)
stacje.to_csv('wyniki/POMOCNICZE_TABELE/stacje.csv', index=False)

In [77]:
#ZIMA - STYCZEN

mapa_styczen = folium.Map(location=[40.720, -74.040], zoom_start=14)

min_przejazdow = trasy_styczen['liczba_tras'].min()
max_przejazdow = trasy_styczen['liczba_tras'].max()

skala_kolorow = cm.LinearColormap(
    colors=['#ff9999', '#ff0000', '#8b0000'],
    vmin=min_przejazdow, 
    vmax=max_przejazdow,
    caption='Liczba przejazdów',
)
skala_kolorow.add_to(mapa_styczen)

uzywane_stacje = set(trasy_styczen['start_station_name']).union(set(trasy_styczen['end_station_name']))

#dodajemy wszystkie pinezki stacji
for nazwa in uzywane_stacje:
    znaleziona_stacja = stacje[stacje['nazwa_stacji'] == nazwa]
    
    if not znaleziona_stacja.empty:
        lat = znaleziona_stacja.iloc[0]['szerokosc_geo']
        lon = znaleziona_stacja.iloc[0]['dlugosc_geo']
        
        folium.Marker(
            location=[lat, lon],
            popup=nazwa,
            icon=folium.Icon(color='blue', icon='bicycle', prefix='fa')
        ).add_to(mapa_styczen)

# dodajemy odcinki laczace te trasy
for index, row in trasy_styczen.iterrows():
    start = row['start_station_name']
    koniec = row['end_station_name']
    liczba_przejazdow = row['liczba_tras']
    
    znaleziony_start = stacje[stacje['nazwa_stacji'] == start]
    znaleziony_koniec = stacje[stacje['nazwa_stacji'] == koniec]
    
    if not znaleziony_start.empty and not znaleziony_koniec.empty:
        start_coords = [znaleziony_start.iloc[0]['szerokosc_geo'], znaleziony_start.iloc[0]['dlugosc_geo']]
        end_coords = [znaleziony_koniec.iloc[0]['szerokosc_geo'], znaleziony_koniec.iloc[0]['dlugosc_geo']]
        
        grubosc_linii = max(1, liczba_przejazdow / 75) 
        kolor_linii = skala_kolorow(liczba_przejazdow)
        
        folium.PolyLine(
            locations=[start_coords, end_coords],
            color=kolor_linii,
            weight=grubosc_linii,
            opacity=0.8,
            tooltip=f"{start} -> {koniec} ({liczba_przejazdow} przejazdów)"
        ).add_to(mapa_styczen)

mapa_styczen.save('WYNIKI/MAPY/mapa_styczen.html')

In [78]:
# WIOSNA - KWIECIEN

mapa_kwiecien = folium.Map(location=[40.720, -74.040], zoom_start=14)

min_przejazdow = trasy_kwiecien['liczba_tras'].min()
max_przejazdow = trasy_kwiecien['liczba_tras'].max()

skala_kolorow = cm.LinearColormap(
    colors=['#ff9999', '#ff0000', '#8b0000'],
    vmin=min_przejazdow, 
    vmax=max_przejazdow,
    caption='Liczba przejazdów',
)
skala_kolorow.add_to(mapa_kwiecien)

uzywane_stacje = set(trasy_kwiecien['start_station_name']).union(set(trasy_kwiecien['end_station_name']))

#dodajemy wszystkie pinezki stacji
for nazwa in uzywane_stacje:
    znaleziona_stacja = stacje[stacje['nazwa_stacji'] == nazwa]
    
    if not znaleziona_stacja.empty:
        lat = znaleziona_stacja.iloc[0]['szerokosc_geo']
        lon = znaleziona_stacja.iloc[0]['dlugosc_geo']
        
        folium.Marker(
            location=[lat, lon],
            popup=nazwa,
            icon=folium.Icon(color='blue', icon='bicycle', prefix='fa')
        ).add_to(mapa_kwiecien)

# dodajemy odcinki laczace te trasy
for index, row in trasy_kwiecien.iterrows():
    start = row['start_station_name']
    koniec = row['end_station_name']
    liczba_przejazdow = row['liczba_tras']
    
    znaleziony_start = stacje[stacje['nazwa_stacji'] == start]
    znaleziony_koniec = stacje[stacje['nazwa_stacji'] == koniec]
    
    if not znaleziony_start.empty and not znaleziony_koniec.empty:
        start_coords = [znaleziony_start.iloc[0]['szerokosc_geo'], znaleziony_start.iloc[0]['dlugosc_geo']]
        end_coords = [znaleziony_koniec.iloc[0]['szerokosc_geo'], znaleziony_koniec.iloc[0]['dlugosc_geo']]
        
        grubosc_linii = max(1, liczba_przejazdow / 75) 
        kolor_linii = skala_kolorow(liczba_przejazdow)
        
        folium.PolyLine(
            locations=[start_coords, end_coords],
            color=kolor_linii,
            weight=grubosc_linii,
            opacity=0.8,
            tooltip=f"{start} -> {koniec} ({liczba_przejazdow} przejazdów)"
        ).add_to(mapa_kwiecien)

mapa_kwiecien.save('WYNIKI/MAPY/mapa_kwiecien.html')

In [79]:
# LATO - LIPIEC

mapa_lipiec = folium.Map(location=[40.720, -74.040], zoom_start=14)

min_przejazdow = trasy_lipiec['liczba_tras'].min()
max_przejazdow = trasy_lipiec['liczba_tras'].max()

skala_kolorow = cm.LinearColormap(
    colors=['#ff9999', '#ff0000', '#8b0000'],
    vmin=min_przejazdow, 
    vmax=max_przejazdow,
    caption='Liczba przejazdów',
)
skala_kolorow.add_to(mapa_lipiec)

uzywane_stacje = set(trasy_lipiec['start_station_name']).union(set(trasy_lipiec['end_station_name']))

#dodajemy wszystkie pinezki stacji
for nazwa in uzywane_stacje:
    znaleziona_stacja = stacje[stacje['nazwa_stacji'] == nazwa]
    
    if not znaleziona_stacja.empty:
        lat = znaleziona_stacja.iloc[0]['szerokosc_geo']
        lon = znaleziona_stacja.iloc[0]['dlugosc_geo']
        
        folium.Marker(
            location=[lat, lon],
            popup=nazwa,
            icon=folium.Icon(color='blue', icon='bicycle', prefix='fa')
        ).add_to(mapa_lipiec)

# dodajemy odcinki laczace te trasy
for index, row in trasy_lipiec.iterrows():
    start = row['start_station_name']
    koniec = row['end_station_name']
    liczba_przejazdow = row['liczba_tras']
    
    znaleziony_start = stacje[stacje['nazwa_stacji'] == start]
    znaleziony_koniec = stacje[stacje['nazwa_stacji'] == koniec]
    
    if not znaleziony_start.empty and not znaleziony_koniec.empty:
        start_coords = [znaleziony_start.iloc[0]['szerokosc_geo'], znaleziony_start.iloc[0]['dlugosc_geo']]
        end_coords = [znaleziony_koniec.iloc[0]['szerokosc_geo'], znaleziony_koniec.iloc[0]['dlugosc_geo']]
        
        grubosc_linii = max(1, liczba_przejazdow / 75) 
        kolor_linii = skala_kolorow(liczba_przejazdow)
        
        folium.PolyLine(
            locations=[start_coords, end_coords],
            color=kolor_linii,
            weight=grubosc_linii,
            opacity=0.8,
            tooltip=f"{start} -> {koniec} ({liczba_przejazdow} przejazdów)"
        ).add_to(mapa_lipiec)

mapa_lipiec.save('WYNIKI/MAPY/mapa_lipiec.html')

In [80]:
# JESIEN - PAZDZIERNIK

mapa_pazdziernik = folium.Map(location=[40.720, -74.040], zoom_start=14)

min_przejazdow = trasy_pazdziernik['liczba_tras'].min()
max_przejazdow = trasy_pazdziernik['liczba_tras'].max()

skala_kolorow = cm.LinearColormap(
    colors=['#ff9999', '#ff0000', '#8b0000'],
    vmin=min_przejazdow, 
    vmax=max_przejazdow,
    caption='Liczba przejazdów',
)
skala_kolorow.add_to(mapa_pazdziernik)

uzywane_stacje = set(trasy_pazdziernik['start_station_name']).union(set(trasy_pazdziernik['end_station_name']))

#dodajemy wszystkie pinezki stacji
for nazwa in uzywane_stacje:
    znaleziona_stacja = stacje[stacje['nazwa_stacji'] == nazwa]
    
    if not znaleziona_stacja.empty:
        lat = znaleziona_stacja.iloc[0]['szerokosc_geo']
        lon = znaleziona_stacja.iloc[0]['dlugosc_geo']
        
        folium.Marker(
            location=[lat, lon],
            popup=nazwa,
            icon=folium.Icon(color='blue', icon='bicycle', prefix='fa')
        ).add_to(mapa_pazdziernik)

# dodajemy odcinki laczace te trasy
for index, row in trasy_pazdziernik.iterrows():
    start = row['start_station_name']
    koniec = row['end_station_name']
    liczba_przejazdow = row['liczba_tras']
    
    znaleziony_start = stacje[stacje['nazwa_stacji'] == start]
    znaleziony_koniec = stacje[stacje['nazwa_stacji'] == koniec]
    
    if not znaleziony_start.empty and not znaleziony_koniec.empty:
        start_coords = [znaleziony_start.iloc[0]['szerokosc_geo'], znaleziony_start.iloc[0]['dlugosc_geo']]
        end_coords = [znaleziony_koniec.iloc[0]['szerokosc_geo'], znaleziony_koniec.iloc[0]['dlugosc_geo']]
        
        grubosc_linii = max(1, liczba_przejazdow / 75) 
        kolor_linii = skala_kolorow(liczba_przejazdow)
        
        folium.PolyLine(
            locations=[start_coords, end_coords],
            color=kolor_linii,
            weight=grubosc_linii,
            opacity=0.8,
            tooltip=f"{start} -> {koniec} ({liczba_przejazdow} przejazdów)"
        ).add_to(mapa_pazdziernik)

mapa_pazdziernik.save('WYNIKI/MAPY/mapa_pazdziernik.html')